In [1]:
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import seaborn as sns

print("UCITAVANJE PODATAKA")

df = pd.read_csv('../../archive/yellow_tripdata_2015-01.csv')
print(f"Ucitano: {len(df):,} redova, {len(df.columns)} kolona")

UCITAVANJE PODATAKA
Ucitano: 12,748,986 redova, 19 kolona


In [2]:
print("PROVERA NEDOSTAJUCIH VREDNOSTI")

missing_values = df.isnull().sum()
missing_cols = missing_values[missing_values > 0]
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("Nema nedostajucih vrednosti!")

PROVERA NEDOSTAJUCIH VREDNOSTI
improvement_surcharge    3
dtype: int64


In [3]:
print("\n" + "="*60)
print("PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)")
print("="*60)

zero_distance = df[df['trip_distance'] == 0]
print(f"Vožnje sa 0 milja: {len(zero_distance):,}")

long_distance = df[df['trip_distance'] > 100]
print(f"Vožnje preko 100 milja: {len(long_distance):,}")

zero_fare = df[df['fare_amount'] <= 0]
print(f"Vožnje sa iznosom <= 0: {len(zero_fare):,}")

high_fare = df[df['fare_amount'] > 500]
print(f"Vožnje sa iznosom > 500$: {len(high_fare):,}")

invalid_passengers = df[(df['passenger_count'] == 0) | (df['passenger_count'] > 6)]
print(f"Vožnje sa neispravnim brojem putnika (0 ili >6): {len(invalid_passengers):,}")

negative_tip = df[df['tip_amount'] < 0]
print(f"Vožnje sa negativnom napojnicom: {len(negative_tip):,}")


PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)
Vožnje sa 0 milja: 79,365
Vožnje preko 100 milja: 172
Vožnje sa iznosom <= 0: 7,666
Vožnje sa iznosom > 500$: 77
Vožnje sa neispravnim brojem putnika (0 ili >6): 6,595
Vožnje sa negativnom napojnicom: 79


In [4]:
print("PROVERA VREMENSKIH NEKONZISTENCIJA")

# Konverzija vremena
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Provera dropoff pre pickup
invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]
print(f"Vožnje sa dropoff pre pickup: {len(invalid_time):,}")

# Trajanje vožnje
df['duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

long_rides = df[df['duration_minutes'] > 1440]  # 24h = 1440 min
print(f"Vožnje duže od 24h: {len(long_rides):,}")

zero_duration = df[df['duration_minutes'] == 0]
print(f"Vožnje koje traju 0 minuta: {len(zero_duration):,}")

PROVERA VREMENSKIH NEKONZISTENCIJA
Vožnje sa dropoff pre pickup: 297
Vožnje duže od 24h: 79
Vožnje koje traju 0 minuta: 14,816


In [5]:
print("OSNOVNO CISCENJE")
print(f"Pre ciscenja: {len(df):,}")

df_clean = df[
    # Vreme
    (df['tpep_dropoff_datetime'] >= df['tpep_pickup_datetime']) &
    (df['duration_minutes'] > 0) &
    (df['duration_minutes'] <= 1440) &
    
    # Distanca
    (df['trip_distance'] > 0) &
    (df['trip_distance'] <= 100) &
    
    # Cena
    (df['fare_amount'] > 0) &
    (df['fare_amount'] <= 500) &
    (df['tip_amount'] >= 0) &
    
    # Putnici
    (df['passenger_count'] >= 1) &
    (df['passenger_count'] <= 6)
].copy()

print(f"Posle osnovnog ciscenja: {len(df_clean):,}")
print(f"Uklonjeno: {len(df) - len(df_clean):,} redova ({((len(df)-len(df_clean))/len(df)*100):.2f}%)")

# Oslobadjam memoriju
del df
gc.collect()

OSNOVNO CISCENJE
Pre ciscenja: 12,748,986
Posle osnovnog ciscenja: 12,657,281
Uklonjeno: 91,705 redova (0.72%)


0

In [6]:
print("POPUNJAVANJE NULL VREDNOSTI")
print(f"NULL pre popunjavanja:")
null_counts = df_clean.isnull().sum()
print(null_counts[null_counts > 0])

# Popuni NULL vrednosti
df_clean['improvement_surcharge'] = df_clean['improvement_surcharge'].fillna(0)
df_clean['store_and_fwd_flag'] = df_clean['store_and_fwd_flag'].fillna('N')

print(f"NULL posle popunjavanja: {df_clean.isnull().sum().sum()}")

POPUNJAVANJE NULL VREDNOSTI
NULL pre popunjavanja:
improvement_surcharge    3
dtype: int64
NULL posle popunjavanja: 0


In [8]:
print("FILTRIRANJE KOORDINATA (NYC OPSEG)")
before = len(df_clean)

df_clean = df_clean[
    (df_clean['pickup_longitude'].between(-74.5, -73.5)) &
    (df_clean['pickup_latitude'].between(40.5, 41.0)) &
    (df_clean['dropoff_longitude'].between(-74.5, -73.5)) &
    (df_clean['dropoff_latitude'].between(40.5, 41.0))
]

print(f"Uklonjeno van NYC opsega: {before - len(df_clean):,} redova")

FILTRIRANJE KOORDINATA (NYC OPSEG)
Uklonjeno van NYC opsega: 234,683 redova


In [9]:
print("UKLANJANJE OUTLIER-A (IQR METODA)")
def remove_outliers_iqr(df, column, multiplier=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    
    before = len(df)
    df_clean = df[(df[column] >= lower) & (df[column] <= upper)]
    
    print(f"  {column}:")
    print(f"    Opseg: {lower:.2f} - {upper:.2f}")
    print(f"    Uklonjeno: {before - len(df_clean):,} redova")
    
    return df_clean

outlier_cols = ['trip_distance', 'fare_amount', 'total_amount']
for col in outlier_cols:
    df_clean = remove_outliers_iqr(df_clean, col)

UKLANJANJE OUTLIER-A (IQR METODA)
  trip_distance:
    Opseg: -2.01 - 6.02
    Uklonjeno: 1,231,005 redova
  fare_amount:
    Opseg: -2.25 - 19.75
    Uklonjeno: 277,301 redova
  total_amount:
    Opseg: -1.20 - 22.80
    Uklonjeno: 177,110 redova


In [10]:
print("FILTRIRANJE KATEGORIJALNIH VARIJABLI")
before = len(df_clean)

df_clean = df_clean[df_clean['VendorID'].isin([1, 2])]
df_clean = df_clean[df_clean['RateCodeID'].isin([1, 2, 3, 4, 5, 6])]
df_clean = df_clean[df_clean['payment_type'].isin([1, 2, 3, 4, 5, 6])]

print(f"Uklonjeno sa nevalidnim kodovima: {before - len(df_clean):,}")

FILTRIRANJE KATEGORIJALNIH VARIJABLI
Uklonjeno sa nevalidnim kodovima: 195


In [11]:
print("UKLANJANJE DUPLIKATA")
before = len(df_clean)

# Samo potpuni duplikati (isti pickup, dropoff, vreme)
df_clean = df_clean.drop_duplicates(
    subset=['pickup_longitude', 'pickup_latitude', 
            'dropoff_longitude', 'dropoff_latitude',
            'tpep_pickup_datetime', 'tpep_dropoff_datetime']
)

print(f"Uklonjeno potpunih duplikata: {before - len(df_clean):,}")


UKLANJANJE DUPLIKATA
Uklonjeno potpunih duplikata: 0


In [12]:
# Brzina (milja po satu)
df_clean['avg_speed_mph'] = df_clean['trip_distance'] / (df_clean['duration_minutes'] / 60)

# Cena po milji
df_clean['price_per_mile'] = df_clean['fare_amount'] / df_clean['trip_distance']

# Vremenske karakteristike
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour
df_clean['pickup_day'] = df_clean['tpep_pickup_datetime'].dt.dayofweek
df_clean['pickup_month'] = df_clean['tpep_pickup_datetime'].dt.month
df_clean['pickup_day_name'] = df_clean['tpep_pickup_datetime'].dt.day_name()
df_clean['is_weekend'] = (df_clean['pickup_day'] >= 5).astype(int)

# Rush hour  (7-9 i 16-19)
df_clean['is_rush_hour'] = df_clean['pickup_hour'].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

print(f"Dodato 7 novih karakteristika")
print(f"Ukupno kolona: {len(df_clean.columns)}")


Dodato 7 novih karakteristika
Ukupno kolona: 28


In [14]:
print("KONAČAN BROJ REDOVA I KOLONA")
print(f"Broj redova: {len(df_clean):,}")
print(f"Broj kolona: {len(df_clean.columns)}")

KONAČAN BROJ REDOVA I KOLONA
Broj redova: 10,736,987
Broj kolona: 28


In [15]:
print("CUVANJE OCISCENIH PODATAKA")
putanja = r'C:\Users\Andjela\Desktop\yellow_tripdata_2015-01_OCISCEN.csv'
df_clean.to_csv(putanja, index=False)
print(f"Fajl sacuvan na: {putanja}")
print("CISCENJE ZAVRSENO!")

CUVANJE OCISCENIH PODATAKA
Fajl sacuvan na: C:\Users\Andjela\Desktop\yellow_tripdata_2015-01_OCISCEN.csv
CISCENJE ZAVRSENO!


In [5]:
import pandas as pd
from sqlalchemy import create_engine
import pyodbc

server = 'DESKTOP-64J7LMV\\MSSQLSERVER2022' 
database = 'SkladistenjePROJEKAT'           

engine = create_engine(
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes'
)

try:
    with engine.connect() as conn:
        print("Povezano!")
except Exception as e:
    print(f"Greška! {e}")



Povezano!
